# DeepPEF Evolution Training Pipeline (Colab)

**Levels of experiments — each builds on what the previous taught us.**

| Level | What | Expected PCC | Time (A100) |
|-------|------|-------------|-------------|
| 7 | ProtT5 baseline 5-seed ensemble | 0.52-0.57 | ~3-4 hrs |
| 9 | dual_esmif + projection 5-seed | 0.53-0.56 | ~3-4 hrs |
| 10 | GNN-SM (subtract-mut, novel) | 0.50-0.65 | ~2-3 hrs |
| MLP | MLP subtract-mut baseline | 0.30-0.50 | ~15 min |

**TOTAL ESTIMATED TIME: ~9-11 hours on A100 (run overnight)**

---

## Prerequisites

1. Select A100 or V100 runtime (Premium Colab)
2. Run all cells — data downloads automatically from HuggingFace

In [ ]:
# Cell 1: Mount Google Drive (for saving results/models later)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/DeepPEF_data', exist_ok=True)
print('Drive mounted. Results will be saved to: My Drive/DeepPEF_data/')

In [ ]:
# Cell 2: Clone repo and install ALL dependencies
!git clone https://github.com/nissimbrami/DeepEF-Thesis.git /content/DeepPEF
%cd /content/DeepPEF

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"}')

# Install PyG with correct CUDA version
CUDA_VERSION = torch.version.cuda.replace('.', '')
TORCH_VERSION = torch.__version__.split('+')[0]
print(f'Installing PyG for torch={TORCH_VERSION}, cuda={CUDA_VERSION}')

!pip install torch-geometric -q
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION[:3]}.html -q 2>/dev/null || \
    pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html -q 2>/dev/null || \
    echo 'PyG extra libs install had issues - continuing anyway'

!pip install scipy scikit-learn pandas tqdm biopython wandb -q

# Verify PyG works
try:
    from torch_geometric.nn import GCNConv, GATv2Conv
    print('\nPyTorch Geometric: OK')
except ImportError as e:
    print(f'\nWARNING: PyG import failed: {e}')
    print('Trying alternative install...')
    !pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu118.html -q
    from torch_geometric.nn import GCNConv, GATv2Conv
    print('PyTorch Geometric: OK (after retry)')

In [ ]:
# Cell 3: Download data from HuggingFace (READ ONLY)
!pip install huggingface_hub -q

from huggingface_hub import snapshot_download
import os, shutil

# Check disk space
stat = shutil.disk_usage('/content')
free_gb = stat.free / (1024**3)
print(f'Available disk: {free_gb:.1f} GB')
if free_gb < 80:
    print(f'WARNING: Only {free_gb:.1f}GB free. May be tight for 75GB dataset.')
    print('Consider using A100 runtime which has more disk space.')

print('Downloading data from HuggingFace (nissimb/deepef-megascale)...')
print('This may take 10-30 min depending on connection speed.')

# Download full dataset to local disk
local_path = snapshot_download(
    repo_id='nissimb/deepef-megascale',
    repo_type='dataset',
    local_dir='/content/DeepPEF/hf_data',
)
print(f'Downloaded to: {local_path}')

# Create proper directory structure with symlinks
os.makedirs('./data/MsDs/training_data', exist_ok=True)
os.makedirs('./data/MsDs', exist_ok=True)
os.makedirs('./data', exist_ok=True)

# Determine where protein folders are
hf_root = '/content/DeepPEF/hf_data'

if os.path.exists(os.path.join(hf_root, 'data', 'MsDs', 'training_data', '1A0N')):
    # Structure A: properly nested under data/MsDs/training_data/
    os.rmdir('./data/MsDs/training_data')
    os.symlink(os.path.join(hf_root, 'data', 'MsDs', 'training_data'), './data/MsDs/training_data')
    print('Linked training_data (nested structure)')
else:
    # Structure B: protein folders at root level of HF repo
    # Must create individual symlinks to avoid including 'data/' dir as a protein
    protein_count = 0
    for item in os.listdir(hf_root):
        item_path = os.path.join(hf_root, item)
        # Only link actual protein folders (have .pt files inside, skip 'data' and hidden dirs)
        if os.path.isdir(item_path) and item not in ('data', '.cache', '.huggingface'):
            if item.startswith('.'):
                continue
            # Check if it looks like a protein folder (has coords.pt or emb.pt)
            if os.path.exists(os.path.join(item_path, 'coords.pt')) or \
               os.path.exists(os.path.join(item_path, 'emb.pt')):
                target = os.path.join('./data/MsDs/training_data', item)
                if not os.path.exists(target):
                    os.symlink(item_path, target)
                protein_count += 1
    print(f'Linked {protein_count} protein folders individually')
    if protein_count == 0:
        print('ERROR: No protein folders found! Listing hf_data contents:')
        for item in sorted(os.listdir(hf_root))[:30]:
            print(f'  {item}')
        raise FileNotFoundError('No protein folders found in HuggingFace download!')

# Link mutation_files
if os.path.exists(os.path.join(hf_root, 'data', 'MsDs', 'mutation_files')):
    if not os.path.exists('./data/MsDs/mutation_files'):
        os.symlink(os.path.join(hf_root, 'data', 'MsDs', 'mutation_files'), './data/MsDs/mutation_files')
    print('Linked mutation_files')

# Link ThermoMPNN
if os.path.exists(os.path.join(hf_root, 'data', 'ThermoMPNN')):
    if not os.path.exists('./data/ThermoMPNN'):
        os.symlink(os.path.join(hf_root, 'data', 'ThermoMPNN'), './data/ThermoMPNN')
    print('Linked ThermoMPNN')

# Link Pnas_filtering
if os.path.exists(os.path.join(hf_root, 'data', 'Processed_K50_dG_datasets')):
    os.makedirs('./data/Processed_K50_dG_datasets', exist_ok=True)
    pnas_src = os.path.join(hf_root, 'data', 'Processed_K50_dG_datasets', 'Pnas_filtering')
    if os.path.exists(pnas_src) and not os.path.exists('./data/Processed_K50_dG_datasets/Pnas_filtering'):
        os.symlink(pnas_src, './data/Processed_K50_dG_datasets/Pnas_filtering')
    print('Linked Pnas_filtering')

# Final verification
print(f'\nData structure:')
print(f'  Training data: {os.path.exists("./data/MsDs/training_data")} ({len(os.listdir("./data/MsDs/training_data"))} items)')
print(f'  Mutation files: {os.path.exists("./data/MsDs/mutation_files")}')
print(f'  ThermoMPNN: {os.path.exists("./data/ThermoMPNN")}')
print(f'  Pnas_filtering: {os.path.exists("./data/Processed_K50_dG_datasets/Pnas_filtering")}')

In [ ]:
# Cell 4: Verify ALL data integrity (MUST pass before training)
import torch, os
import pandas as pd

td = './data/MsDs/training_data'
mut_dir = './data/MsDs/mutation_files'

# Check proteins
proteins = [d for d in os.listdir(td) if os.path.isdir(os.path.join(td, d))]
print(f'Total proteins: {len(proteins)}')
assert len(proteins) >= 300, f'FATAL: Only {len(proteins)} proteins! Expected 368'

# Check critical files
missing_coords = [p for p in proteins if not os.path.exists(os.path.join(td, p, 'coords.pt'))]
missing_emb = [p for p in proteins if not os.path.exists(os.path.join(td, p, 'emb.pt'))]
missing_mask = [p for p in proteins if not os.path.exists(os.path.join(td, p, 'mask.pt'))]
missing_dg = [p for p in proteins if not os.path.exists(os.path.join(td, p, 'deltaG.pt'))]
missing_esmif = [p for p in proteins if not os.path.exists(os.path.join(td, p, 'esmif_enc.pt'))]

print(f'Missing coords.pt: {len(missing_coords)}')
print(f'Missing emb.pt: {len(missing_emb)}')
print(f'Missing mask.pt: {len(missing_mask)}')
print(f'Missing deltaG.pt: {len(missing_dg)}')
print(f'Missing esmif_enc.pt: {len(missing_esmif)}')
assert len(missing_coords) == 0, f'FATAL: {len(missing_coords)} proteins missing coords!'
assert len(missing_emb) == 0, f'FATAL: {len(missing_emb)} proteins missing embeddings!'

# Check mutation files
missing_mut = [p for p in proteins if not os.path.exists(os.path.join(mut_dir, f'{p}.csv'))]
print(f'Missing mutation CSVs: {len(missing_mut)}/{len(proteins)}')
assert len(missing_mut) < len(proteins) * 0.5, f'FATAL: Too many missing mutation files!'

# Check test split
tm = pd.read_csv('./data/ThermoMPNN/mega_test.csv')
test_names = tm['WT_name'].str.replace('.pdb', '', regex=False).unique().tolist()
test_found = [p for p in test_names if p in proteins]
print(f'Test proteins: {len(test_found)}/{len(test_names)}')
assert len(test_found) > 0, 'FATAL: 0 test proteins found!'

# Check PNAS filtering files
assert os.path.exists('./data/Processed_K50_dG_datasets/Pnas_filtering/train_proteins.csv'), 'Missing train_proteins.csv!'
assert os.path.exists('./data/Processed_K50_dG_datasets/Pnas_filtering/pnas_mutations.csv'), 'Missing pnas_mutations.csv!'
assert os.path.exists('./data/ThermoMPNN/mega_train.csv'), 'Missing mega_train.csv!'

# Check shapes of a sample protein
sample = proteins[0]
c = torch.load(os.path.join(td, sample, 'coords.pt'), weights_only=False)
e = torch.load(os.path.join(td, sample, 'emb.pt'), weights_only=False)
print(f'\nSample: {sample}')
print(f'  coords: {c.shape}')
if isinstance(e, list):
    print(f'  emb: list[{len(e)}], first: {e[0].shape}')
else:
    print(f'  emb: {e.shape}')

# GPU info
print(f'\nGPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
print(f'\n=== ALL CHECKS PASSED â€” READY TO TRAIN ===')

In [ ]:
# Cell 5: Setup environment variables and results tracking
import os, json, time
os.environ['WANDB_MODE'] = 'disabled'
os.environ['PYTHONUNBUFFERED'] = '1'

# Results collector
RESULTS_FILE = '/content/DeepPEF/results_summary.json'
results = {'level7': {}, 'level9': {}, 'level10': {}, 'mlp': {}}

def save_results():
    with open(RESULTS_FILE, 'w') as f:
        json.dump(results, f, indent=2)
    # Also save to Drive for persistence
    drive_path = '/content/drive/MyDrive/DeepPEF_data/results_summary.json'
    with open(drive_path, 'w') as f:
        json.dump(results, f, indent=2)

print('Results will be saved to:', RESULTS_FILE)
print('And backed up to Google Drive')
start_time = time.time()

---
## Level 7 â€” ProtT5 Baseline (5-seed ensemble)
Known-good config: PCC ~0.52 per seed, ensemble ~0.55-0.57

**What we learn:** Establishes the floor. Everything else must beat this.

In [ ]:
%%time
# Level 7: ProtT5 baseline — 5 seeds (output logged)
import os
os.makedirs('logs', exist_ok=True)
seeds = [42, 123, 456, 789, 1337]

for seed in seeds:
    print(f'\n{"="*60}')
    print(f'LEVEL 7: ProtT5 baseline — seed {seed}')
    print(f'{"="*60}')
    !python Megascale-fineTuning/pnas_train.py \
        --model_name baseline_prott5_seed{seed} \
        --seed {seed} \
        --dataset_type pnas \
        --epochs 15 \
        --no_pretrained \
        --loss_type huber_rank \
        --ranking_weight 0.1 \
        --use_knn_gat \
        --one_mut \
        --dg_ml \
        --cosine_lr \
        --lr_min 1e-6 \
        --weight_decay 1e-5 \
        --emb_type prott5 \
        --mini_batch_size 64 \
        --emb_projection none 2>&1 | tee logs/level7_seed{seed}.log
    print(f'Seed {seed} DONE')


In [ ]:
# Save Level 7 models to Drive immediately (in case of disconnect)
import os, shutil
model_dir = './Megascale-fineTuning/models'
if os.path.exists(model_dir):
    !tar czf /content/drive/MyDrive/DeepPEF_data/models_level7.tar.gz Megascale-fineTuning/models/baseline_prott5_*
    print('Level 7 models saved to Drive')
    # Count completed
    l7_models = [d for d in os.listdir(model_dir) if d.startswith('baseline_prott5_')]
    print(f'Completed: {len(l7_models)}/5 seeds')

---
## Level 10 â€” GNN-SM: Subtract-Mut (THE NOVEL APPROACH)

**This is the thesis contribution.** Running BEFORE Level 9 because it's more important.
- Output: [L, 20] amino acid scores (not scalar energy)
- ddG = score[pos, mut_aa] - score[pos, wt_aa]
- 1 forward pass per protein (not 200+)
- Anti-symmetry is AUTOMATIC

**What we learn:** Does full GNN context improve position-specific scoring?

In [ ]:
# Verify GNN-SM works before committing to full training
print('=== GNN-SM VERIFICATION ===')
!python Megascale-fineTuning/pnas_train_sm.py --verify_only --seed 42 --emb_type prott5 --emb_projection none
print('\n=== If you see "VERIFICATION PASSED" above, proceed ===')
print('=== If not, STOP and debug ===')


In [ ]:
%%time
# Level 10: GNN-SM — 5 seeds (30 epochs each)
for seed in seeds:
    print(f'\n{"="*60}')
    print(f'LEVEL 10: GNN-SM (subtract-mut) — seed {seed}')
    print(f'{"="*60}')
    !python Megascale-fineTuning/pnas_train_sm.py \
        --model_name gnn_sm_prott5_seed{seed} \
        --seed {seed} \
        --epochs 30 \
        --emb_type prott5 \
        --emb_projection none \
        --use_knn_gat 2>&1 | tee logs/level10_seed{seed}.log
    print(f'Seed {seed} DONE')


In [ ]:
# Save Level 10 models to Drive
import os
model_dir = './Megascale-fineTuning/models'
if os.path.exists(model_dir):
    !tar czf /content/drive/MyDrive/DeepPEF_data/models_level10.tar.gz Megascale-fineTuning/models/gnn_sm_*
    print('Level 10 models saved to Drive')
    l10_models = [d for d in os.listdir(model_dir) if d.startswith('gnn_sm_')]
    print(f'Completed: {len(l10_models)}/5 seeds')

---
## Level 9 â€” dual_esmif + MLP Projection (5-seed)
ProtT5 (1024) + ESM-IF1 (512) = 1536 dim, projected to 16 before GNN.

**What we learn:** Does structural information (ESM-IF1) add value on top of sequence (ProtT5)?

In [ ]:
%%time
# Level 9: dual_esmif with MLP projection — 5 seeds
for seed in seeds:
    print(f'\n{"="*60}')
    print(f'LEVEL 9: dual_esmif + projection — seed {seed}')
    print(f'{"="*60}')
    !python Megascale-fineTuning/pnas_train.py \
        --model_name v4_dual_esmif_proj_seed{seed} \
        --seed {seed} \
        --dataset_type pnas \
        --epochs 15 \
        --no_pretrained \
        --loss_type huber_rank \
        --ranking_weight 0.1 \
        --use_knn_gat \
        --one_mut \
        --dg_ml \
        --cosine_lr \
        --lr_min 1e-6 \
        --weight_decay 1e-5 \
        --emb_type dual_esmif \
        --mini_batch_size 64 \
        --emb_projection mlp 2>&1 | tee logs/level9_seed{seed}.log
    print(f'Seed {seed} DONE')


In [ ]:
# Save Level 9 models to Drive
import os
model_dir = './Megascale-fineTuning/models'
if os.path.exists(model_dir):
    !tar czf /content/drive/MyDrive/DeepPEF_data/models_level9.tar.gz Megascale-fineTuning/models/v4_dual_esmif_*
    print('Level 9 models saved to Drive')
    l9_models = [d for d in os.listdir(model_dir) if d.startswith('v4_dual_esmif_')]
    print(f'Completed: {len(l9_models)}/5 seeds')

---
## MLP Subtract-Mut Comparison
Simple MLP on frozen ESM-IF1 features â€” shows whether GNN adds value over frozen features + MLP.
This is FAST (~15 min total).

In [ ]:
%%time
# MLP subtract-mut
print('=== MLP Subtract-Mut: ESM-IF1 only ===')
!python Megascale-fineTuning/train_subtract_mut.py --seed 42 --epochs 50 --batch_size 256 2>&1 | tee logs/mlp_esmif.log
print()
print('=== MLP Subtract-Mut: Dual (ProtT5 + ESM-IF1) ===')
!python Megascale-fineTuning/train_subtract_mut.py --seed 42 --dual --epochs 50 --batch_size 256 2>&1 | tee logs/mlp_dual.log


---
## RESULTS SUMMARY

This cell parses all training output and gives you the final numbers.

In [ ]:
# ============================================================
# FINAL RESULTS — Automatic PCC extraction from logs
# ============================================================
import os, re, time, json
import numpy as np

def extract_pcc(log_file):
    '''Extract best PCC from a training log file.'''
    if not os.path.exists(log_file):
        return None
    with open(log_file) as f:
        content = f.read()
    patterns = [
        r'Best Pearson Correlation: ([0-9.]+)',
        r'Best Val PCC = ([0-9.]+)',
        r'Best validation PCC: ([0-9.]+)',
        r'FINAL RESULT.*?PCC.*?([0-9]\.[0-9]+)',
        r'Final result: PCC = ([0-9.]+)',
    ]
    for pat in patterns:
        matches = re.findall(pat, content)
        if matches:
            return float(matches[-1])
    return None

print('=' * 70)
print('     DEEPEF EVOLUTION PIPELINE — FINAL RESULTS')
print('=' * 70)
print(f'Total runtime: {(time.time() - start_time) / 3600:.1f} hours')
print()

seeds = [42, 123, 456, 789, 1337]

# Level 7
print('\n' + '=' * 50)
print('LEVEL 7: ProtT5 Baseline')
print('=' * 50)
l7_pccs = []
for seed in seeds:
    pcc = extract_pcc(f'logs/level7_seed{seed}.log')
    if pcc is not None:
        l7_pccs.append(pcc)
        print(f'  Seed {seed}: PCC = {pcc:.4f}')
    else:
        print(f'  Seed {seed}: FAILED or no result')
if l7_pccs:
    print(f'  ─────────────────────────')
    print(f'  Mean: {np.mean(l7_pccs):.4f} ± {np.std(l7_pccs):.4f}')
    print(f'  Best: {max(l7_pccs):.4f}')

# Level 10
print('\n' + '=' * 50)
print('LEVEL 10: GNN-SM — NOVEL APPROACH')
print('=' * 50)
l10_pccs = []
for seed in seeds:
    pcc = extract_pcc(f'logs/level10_seed{seed}.log')
    if pcc is not None:
        l10_pccs.append(pcc)
        print(f'  Seed {seed}: PCC = {pcc:.4f}')
    else:
        print(f'  Seed {seed}: FAILED or no result')
if l10_pccs:
    print(f'  ─────────────────────────')
    print(f'  Mean: {np.mean(l10_pccs):.4f} ± {np.std(l10_pccs):.4f}')
    print(f'  Best: {max(l10_pccs):.4f}')

# Level 9
print('\n' + '=' * 50)
print('LEVEL 9: dual_esmif + projection')
print('=' * 50)
l9_pccs = []
for seed in seeds:
    pcc = extract_pcc(f'logs/level9_seed{seed}.log')
    if pcc is not None:
        l9_pccs.append(pcc)
        print(f'  Seed {seed}: PCC = {pcc:.4f}')
    else:
        print(f'  Seed {seed}: FAILED or no result')
if l9_pccs:
    print(f'  ─────────────────────────')
    print(f'  Mean: {np.mean(l9_pccs):.4f} ± {np.std(l9_pccs):.4f}')
    print(f'  Best: {max(l9_pccs):.4f}')

# MLP
print('\n' + '=' * 50)
print('MLP SUBTRACT-MUT BASELINE')
print('=' * 50)
mlp_esmif = extract_pcc('logs/mlp_esmif.log')
mlp_dual = extract_pcc('logs/mlp_dual.log')
if mlp_esmif: print(f'  ESM-IF1 only:         PCC = {mlp_esmif:.4f}')
if mlp_dual:  print(f'  Dual (ProtT5+ESM-IF1): PCC = {mlp_dual:.4f}')

# COMPARISON TABLE
print('\n')
print('=' * 70)
print('     FINAL COMPARISON TABLE')
print('=' * 70)
print(f'{"Method":<35} {"Mean PCC":<12} {"Best PCC":<12} {"Seeds"}')
print('-' * 70)
if l7_pccs:
    print(f'{"Level 7: ProtT5 baseline":<35} {np.mean(l7_pccs):<12.4f} {max(l7_pccs):<12.4f} {len(l7_pccs)}/5')
if l10_pccs:
    print(f'{"Level 10: GNN-SM (NOVEL)":<35} {np.mean(l10_pccs):<12.4f} {max(l10_pccs):<12.4f} {len(l10_pccs)}/5')
if l9_pccs:
    print(f'{"Level 9: dual_esmif+proj":<35} {np.mean(l9_pccs):<12.4f} {max(l9_pccs):<12.4f} {len(l9_pccs)}/5')
if mlp_esmif:
    print(f'{"MLP ESM-IF1 only":<35} {mlp_esmif:<12.4f} {"-":<12} {"1"}')
if mlp_dual:
    print(f'{"MLP Dual (ProtT5+ESM-IF1)":<35} {mlp_dual:<12.4f} {"-":<12} {"1"}')
print('-' * 70)
print()

# Winner
all_means = {}
if l7_pccs: all_means['Level 7 (baseline)'] = np.mean(l7_pccs)
if l10_pccs: all_means['Level 10 (GNN-SM)'] = np.mean(l10_pccs)
if l9_pccs: all_means['Level 9 (dual)'] = np.mean(l9_pccs)
if all_means:
    winner = max(all_means, key=all_means.get)
    print(f'WINNER: {winner} with mean PCC = {all_means[winner]:.4f}')
print()

# Save to Drive
results_dict = {
    'level7': {'pccs': l7_pccs, 'mean': float(np.mean(l7_pccs)) if l7_pccs else None},
    'level10': {'pccs': l10_pccs, 'mean': float(np.mean(l10_pccs)) if l10_pccs else None},
    'level9': {'pccs': l9_pccs, 'mean': float(np.mean(l9_pccs)) if l9_pccs else None},
    'mlp_esmif': mlp_esmif,
    'mlp_dual': mlp_dual,
}
with open('results_summary.json', 'w') as f:
    json.dump(results_dict, f, indent=2)
try:
    with open('/content/drive/MyDrive/DeepPEF_data/results_summary.json', 'w') as f:
        json.dump(results_dict, f, indent=2)
    print('Results JSON saved to Google Drive')
except:
    print('Could not save to Drive (may be disconnected)')

# Also save logs to Drive
!tar czf /content/drive/MyDrive/DeepPEF_data/training_logs.tar.gz logs/ 2>/dev/null && echo 'Logs saved to Drive' || echo 'Could not save logs'


In [ ]:
# Save ALL models to Drive (final backup)
import os
model_dir = './Megascale-fineTuning/models'
if os.path.exists(model_dir) and os.listdir(model_dir):
    !tar czf /content/drive/MyDrive/DeepPEF_data/all_trained_models.tar.gz Megascale-fineTuning/models/
    size_mb = os.path.getsize('/content/drive/MyDrive/DeepPEF_data/all_trained_models.tar.gz') / 1024**2
    print(f'ALL models saved to Drive ({size_mb:.1f} MB)')
    print('Path: /content/drive/MyDrive/DeepPEF_data/all_trained_models.tar.gz')
else:
    print('No models to save')